# 00 — Z0 Zero-Shot Benchmark & Phase A8/A9 Domain Gap Diagnostics

Evaluates the frozen backbone (`Qwen/Qwen2-VL-2B` primary, `SmolVLM-Base`, `SmolVLM-500M-Base`) on `NEW_validation` + `NEW_test`, generates 2D UMAP projections (A8), and computes RKHS MMD & 5-fold Proxy $\mathcal{A}$-distance (A9).


In [1]:
# ==============================================================================
# 1. COLAB SETUP, REPO SYNC & GPU CHECK (L4 > A100 > G4 > T4 FALLBACK)
# ==============================================================================
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

def run_streamed(cmd):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

if not Path("TRAIN/src").exists():
    if Path("VQA_Final_Thesis/TRAIN/src").exists():
        os.chdir("VQA_Final_Thesis")
    else:
        subprocess.run(["git", "clone", "https://github.com/NhanChanDeo/VQA_Final_Thesis.git"], check=True)
        os.chdir("VQA_Final_Thesis")

subprocess.run(["git", "pull", "origin", "main"], check=True)
print("Working directory:", Path.cwd())

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers>=4.49.0", "peft", "bitsandbytes", "accelerate",
        "datasets", "polars", "pyyaml", "safetensors", "huggingface_hub",
        "hf_transfer", "umap-learn", "scikit-learn", "qwen-vl-utils",
    ],
    check=True,
)

import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    print(f"GPU: {props.name} | VRAM: {vram_gb:.1f} GB | bf16: {torch.cuda.is_bf16_supported()}")
else:
    print("WARNING: No CUDA GPU detected. Switch Colab runtime to L4, A100, G4, or T4.")

Working directory: /content/VQA_Final_Thesis


In [2]:
# ==============================================================================
# 2. MASKED API KEY INPUT BOX & GOOGLE DRIVE CHECKPOINT BACKUP
# ==============================================================================
import os
from getpass import getpass
from pathlib import Path
from huggingface_hub import HfApi

# 2A. Interactive masked HF WRITE token (never stored in notebook source/outputs)
HF_TOKEN = os.environ.get("HF_TOKEN") or None
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    print("Enter your Hugging Face WRITE Token (input is masked and never saved):")
    HF_TOKEN = getpass("HF Write Token: ").strip()

os.environ["HF_TOKEN"] = HF_TOKEN
assert HF_TOKEN, "HF_TOKEN is required to save/restore private checkpoints."

api = HfApi(token=HF_TOKEN)
user_info = api.whoami()
print(f"Authenticated Hugging Face account: {user_info['name']}")

# 2B. Mount Google Drive for dual-redundant checkpoint backup
GDRIVE_BACKUP_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_ckpt_root = Path("/content/drive/MyDrive/CrossVQA_Checkpoints/ALT_THREE_SPLITS")
    drive_ckpt_root.mkdir(parents=True, exist_ok=True)
    GDRIVE_BACKUP_DIR = str(drive_ckpt_root)
    print(f"Google Drive backup active: {GDRIVE_BACKUP_DIR}")
except Exception as e:
    print(f"Google Drive mount skipped ({e}); continuing with private HF Hub checkpoints only.")

Mounted at /content/drive
Google Drive backup active: /content/drive/MyDrive/CrossVQA_Checkpoints/ALT_THREE_SPLITS


In [3]:
# ==============================================================================
# 3. FAST NVMe SNAPSHOT OF `three-splits` SHARDS TO /content/data/
# ==============================================================================
from pathlib import Path
from huggingface_hub import snapshot_download

LOCAL_DATA_DIR = Path("/content/data")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

for ds_name, repo_id in [
    ("docvqa", "gamusa/docvqa-canonical"),
    ("textvqa", "gamusa/textvqa-canonical"),
]:
    target = LOCAL_DATA_DIR / ds_name
    print(f"Syncing {repo_id}@three-splits -> {target} ...")
    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        revision="three-splits",
        allow_patterns=["data/*.parquet"],
        local_dir=str(target),
        token=os.environ.get("HF_TOKEN"),
    )
print("Snapshot ready on ephemeral NVMe scratch.")

Syncing gamusa/docvqa-canonical@three-splits -> /content/data/docvqa ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Syncing gamusa/textvqa-canonical@three-splits -> /content/data/textvqa ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

Snapshot ready on ephemeral NVMe scratch.


In [ ]:
# ==============================================================================
# 4. RUN Z0 ZERO-SHOT + PHASE A8 UMAP + PHASE A9 MMD / PROXY A-DISTANCE
# ==============================================================================
import subprocess
import sys

MODEL_KEY = "qwen2_vl_2b"        # Options: "qwen2_vl_2b", "smolvlm_base", "smolvlm_500m"
HARDWARE_PROFILE = "auto"        # Options: "auto", "l4_24gb", "a100_40gb", "g4_96gb", "t4_16gb"
DOMAIN_GAP_SAMPLES = 2000        # Number of validation samples per domain for UMAP/MMD/Proxy-A
MAX_EVAL_SAMPLES = 20         # Set e.g. 100 for a quick pilot smoke test

cmd = [
    sys.executable, "TRAIN/ALT_THREE_SPLITS/scripts/run_z0_domain_gap.py",
    "--model", MODEL_KEY,
    "--hardware_profile", HARDWARE_PROFILE,
    "--domain_gap_samples", str(DOMAIN_GAP_SAMPLES),
    "--local_data_dir", str(LOCAL_DATA_DIR),
    "--skip_domain_gap",
]
if MAX_EVAL_SAMPLES is not None:
    cmd.extend(["--max_eval_samples", str(MAX_EVAL_SAMPLES)])

print("Running:", " ".join(cmd))
run_streamed(cmd)

Running: /usr/bin/python3 TRAIN/ALT_THREE_SPLITS/scripts/run_z0_domain_gap.py --model qwen2_vl_2b --hardware_profile auto --domain_gap_samples 2000 --local_data_dir /content/data --skip_domain_gap --max_eval_samples 20
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
Loading backbone 'Qwen/Qwen2-VL-2B' (4-bit: False, dtype: torch.float32, attn: sdpa)...

Loading weights: 100%|██████████| 729/729 [00:27<00:00, 26.25it/s]

=== Running Z0 Zero-Shot Evaluation on 'validation' ===

Evaluating:   5%|▌         | 1/20 [00:00<00:09,  2.07it/s]/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_pro

In [ ]:
# ==============================================================================
# 5. INSPECT RESULTS & SYNC LIGHTWEIGHT LOGS TO GITHUB (WEIGHTS EXCLUDED)
# ==============================================================================
import json
import subprocess
from getpass import getpass
from pathlib import Path

# Preview generated metrics/run cards
for p in sorted(Path("TRAIN/ALT_THREE_SPLITS/runs").rglob("val_metrics.json")):
    print("===", p, "===")
    print(p.read_text(encoding="utf-8")[:600])

# Optional masked GitHub push for lightweight JSON/YAML/PNG run artifacts
PUSH_LOGS_TO_GITHUB = True  # Set True when ready to commit logs from Colab
if PUSH_LOGS_TO_GITHUB:
    subprocess.run(["git", "add", "TRAIN/ALT_THREE_SPLITS/runs/"], check=True)
    subprocess.run(["git", "commit", "-m", "chore(runs): sync Colab training metrics and logs"], check=False)
    gh_token = os.environ.get("GH_TOKEN") or getpass("GitHub PAT (masked, optional): ").strip()
    if gh_token:
        remote_url = f"https://{gh_token}@github.com/NhanChanDeo/VQA_Final_Thesis.git"
        subprocess.run(["git", "push", remote_url, "HEAD:main"], check=True)
        print("Synced lightweight logs to origin/main.")

=== TRAIN/ALT_THREE_SPLITS/runs/qwen2_vl_2b/Z0/val_metrics.json ===
{
  "splits": {
    "validation": {
      "docvqa_anls": 0.749893,
      "docvqa_vqa_accuracy": 0.216667,
      "textvqa_anls": 0.838725,
      "textvqa_vqa_accuracy": 0.733333,
      "harmonic_mean": 0.741521
    },
    "test": {
      "docvqa_anls": 0.69398,
      "docvqa_vqa_accuracy": 0.233333,
      "textvqa_anls": 0.9,
      "textvqa_vqa_accuracy": 0.85,
      "harmonic_mean": 0.764107
    }
  },
  "domain_gap": {}
}


## Terminate the session

In [ ]:
from google.colab import runtime
runtime.unassign()